# Capítulo 5 · Árboles de decisión

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/lectures/chapter_05/01_arboles_de_decision.ipynb)

**Aprendizaje supervisado · Parte 1**

---

### ¿Dónde estamos?

En el capítulo 3 aprendimos a **preparar los datos**. Ahora damos el paso que faltaba: **entrenar un modelo, mejorarlo y evaluarlo**. Es el flujo completo de un proyecto de Machine Learning:

```
EDA → limpieza (nulos, duplicados) → train/test → outliers → escalado/codificación   ← ya lo sabes
    → modelo base → optimización de hiperparámetros → evaluación → reporte          ← esta clase
```

**Aprendizaje supervisado** significa que tenemos ejemplos **con la respuesta correcta** (la etiqueta o *target*) y el modelo aprende a predecirla. Si la respuesta es una categoría, es un problema de **clasificación**; si es un número, de **regresión**.

### 🎯 Objetivos
1. Hacer un análisis exploratorio (EDA) con seaborn antes de modelar.
2. Repasar el preprocesamiento: nulos, duplicados, outliers, escalado y codificación.
3. Entender cómo funciona un árbol de decisión.
4. Entrenar un modelo con parámetros por defecto y otro con optimización de hiperparámetros (`GridSearchCV`).
5. Estimar el rendimiento con validación cruzada.
6. Evaluar un clasificador: matriz de confusión, *classification report*, curvas ROC/AUC y curvas de aprendizaje.
7. Resumir los resultados en un reporte.

### 📚 Índice
1. El dataset Iris
2. Análisis exploratorio (EDA)
3. Limpieza: valores nulos y duplicados
4. Variables categóricas
5. Separar train y test
6. Outliers
7. Estandarización
8. ¿Cómo funciona un árbol de decisión?
9. Modelo base (parámetros por defecto) y validación cruzada
10. Optimización de hiperparámetros y comparación con validación cruzada
11. Evaluación de los modelos
12. Curvas de aprendizaje
13. Curvas ROC y AUC
14. Reporte de resultados
15. Chuleta resumen

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn import set_config
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, f1_score,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold, cross_val_score, cross_validate,
                                     learning_curve, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Que scikit-learn devuelva DataFrames (con nombres de columna) en lugar de arrays de NumPy
set_config(transform_output="pandas")
sns.set_theme(style="whitegrid")

# Semilla para que los resultados sean reproducibles
SEED = 42
np.random.seed(SEED)

# Carpeta data/ en la raíz del repo: subimos de carpeta en carpeta hasta dar con pyproject.toml
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"

---
## 1. El dataset Iris

Es el dataset más famoso de Machine Learning (Ronald Fisher, 1936). Tiene **150 flores** de tres especies de iris, y de cada una se midieron 4 cosas en centímetros:

| Variable | Significado |
|---|---|
| `largo_sepalo`, `ancho_sepalo` | Tamaño del sépalo (la "hojita" verde que protege la flor) |
| `largo_petalo`, `ancho_petalo` | Tamaño del pétalo |
| `especie` | **Target**: *setosa*, *versicolor* o *virginica* |

**Nuestra tarea:** a partir de las 4 medidas, predecir la especie. Es un problema de **clasificación multiclase** (3 clases).

> 💡 Si no tienes `data/iris.csv`, ejecuta `uv run python scripts/download_data.py` desde la raíz del repo.

In [ ]:
df = pd.read_csv(DATA_DIR / "iris.csv")

# Nombres de columna más cómodos y la especie como texto en lugar de 0, 1, 2
df.columns = ["largo_sepalo", "ancho_sepalo", "largo_petalo", "ancho_petalo", "especie"]
df["especie"] = df["especie"].map({0: "setosa", 1: "versicolor", 2: "virginica"})

df.head()

---
## 2. Análisis exploratorio (EDA)

Antes de entrenar nada hay que **conocer los datos**. El EDA responde preguntas como: ¿qué tipo tiene cada columna?, ¿hay datos raros?, ¿están las clases equilibradas?, ¿qué variables separan mejor las especies?

### 2.1 Vistazo general

In [ ]:
print("Filas y columnas:", df.shape)
df.info()

In [ ]:
df.describe().round(2)

Las 4 medidas son numéricas y están en la misma unidad (cm), pero con **rangos distintos**: el largo del sépalo va de 4,3 a 7,9 y el ancho del pétalo de 0,1 a 2,5.

### 2.2 ¿Están equilibradas las clases?

Si una clase tuviera el 90 % de las filas, un modelo que siempre dijera esa clase acertaría el 90 % sin aprender nada. Por eso siempre miramos la distribución del *target*.

In [ ]:
ax = sns.countplot(data=df, x="especie", hue="especie", palette="Set2")
ax.set_title("Flores por especie")
plt.show()

df["especie"].value_counts()

✅ **Clases perfectamente equilibradas** (50 de cada una). Así la *accuracy* (porcentaje de aciertos) es una métrica razonable.

### 2.3 Distribución de cada variable por especie

In [ ]:
variables = ["largo_sepalo", "ancho_sepalo", "largo_petalo", "ancho_petalo"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for variable, ax in zip(variables, axes.flat):
    sns.kdeplot(data=df, x=variable, hue="especie", fill=True, palette="Set2", ax=ax)
    ax.set_title(variable)
plt.tight_layout()
plt.show()

Fíjate en las **medidas del pétalo**: la *setosa* (verde) está totalmente separada de las otras dos. *Versicolor* y *virginica* se solapan un poco. En cambio, las medidas del **sépalo** se solapan mucho: por sí solas separan peor las especies.

### 2.4 Relaciones entre variables: `pairplot`

Un `pairplot` dibuja **cada par de variables** en un diagrama de dispersión. Es la forma más rápida de ver qué combinaciones separan las clases.

In [ ]:
sns.pairplot(df, hue="especie", palette="Set2", corner=True)
plt.show()

### 2.5 Correlaciones

La correlación (de -1 a 1) mide si dos variables suben y bajan juntas. Un mapa de calor (`heatmap`) la resume de un vistazo.

In [ ]:
correlaciones = df[variables].corr()

sns.heatmap(correlaciones, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlación entre variables")
plt.show()

El largo y el ancho del pétalo están **muy correlacionados** (0,96): dan casi la misma información. Para un árbol no es un problema, pero conviene saberlo.

### 📝 Conclusiones del EDA
- 150 filas, 4 variables numéricas y un *target* con 3 clases equilibradas.
- Las **medidas del pétalo** son las que mejor separan las especies.
- *Setosa* es fácil de reconocer; la dificultad está en separar *versicolor* de *virginica*.

---
## 3. Limpieza: valores nulos y duplicados

### 3.1 Valores nulos

In [ ]:
df.isna().sum()

✅ **No hay valores nulos.** Aun así, en el `Pipeline` del modelo incluiremos un `SimpleImputer(strategy="median")`. Es una buena costumbre: si mañana llega una flor con una medida faltante, el modelo la rellenará con la mediana de train en lugar de fallar.

> 💡 Si hubiera nulos, **no** los rellenaríamos aquí con la mediana de todo el dataset: eso usaría información del test. El imputador del `Pipeline` calcula la mediana **solo con train**.

### 3.2 Filas duplicadas

In [ ]:
print("Filas duplicadas:", df.duplicated().sum())
df[df.duplicated(keep=False)]

Hay **una fila repetida**. En Iris puede ser una flor real con las mismas medidas (se redondearon a 1 mm), pero la eliminamos igualmente: si una copia acaba en train y otra en test, el modelo "verá" una flor de test durante el entrenamiento y la evaluación será demasiado optimista.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print("Filas tras eliminar duplicados:", len(df))

---
## 4. Variables categóricas

Los modelos solo entienden números. En el capítulo 3 vimos que las variables categóricas **nominales** (sin orden) se codifican con **one-hot encoding**. Veamos si tenemos alguna:

In [ ]:
X = df.drop(columns="especie")
y = df["especie"]

print("Variables categóricas en X:", list(X.select_dtypes(exclude="number").columns))

✅ **Las 4 variables de entrada son numéricas**, así que no hace falta one-hot encoding.

¿Y `especie`? Es categórica, pero es el **target**, no una variable de entrada. Los clasificadores de scikit-learn aceptan el target directamente como texto (`"setosa"`, `"versicolor"`…), así que **no se codifica**.

> 💡 Si el dataset tuviera una variable como `color = rojo / azul / amarillo`, la añadiríamos al `Pipeline` con un `ColumnTransformer` y un `OneHotEncoder(handle_unknown="ignore")`, igual que en el capítulo 3.

---
## 5. Separar train y test

Separamos **antes** de tratar outliers y escalar: así todo lo que "aprendemos" de los datos (límites, medias…) sale solo de train, y el test hace de datos nuevos de verdad.

- `test_size=0.2`: 80 % para entrenar y 20 % para evaluar.
- `stratify=y`: mantiene la proporción de especies en los dos conjuntos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
pd.DataFrame({"train": y_train.value_counts(), "test": y_test.value_counts()})

> ⚠️ El test solo tiene **30 flores**: cada error cuesta más de 3 puntos de *accuracy*. Por eso, para comparar modelos, usaremos también **validación cruzada** en train, que es más estable.

---
## 6. Outliers

### 6.1 Detección con el rango intercuartílico (IQR)

Como en el capítulo 3, un valor es *outlier* si está fuera de $[Q_1 - 1{,}5 \cdot IQR,\; Q_3 + 1{,}5 \cdot IQR]$. Calculamos los límites **solo con train**.

In [ ]:
q1 = X_train.quantile(0.25)
q3 = X_train.quantile(0.75)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

es_outlier = (X_train < limite_inferior) | (X_train > limite_superior)
es_outlier.sum()

In [ ]:
ax = sns.boxplot(data=X_train.melt(var_name="variable", value_name="cm"),
                 x="variable", y="cm", hue="variable", palette="Set2")
ax.set_title("Outliers en train (puntos fuera de los bigotes)")
plt.show()

### 6.2 Tratamiento: recortar (*capping*)

Los pocos outliers están en `ancho_sepalo` y son **valores plausibles** (una flor con un sépalo algo más ancho de lo normal), no errores. Por eso **no eliminamos filas**: con 120 flores en train, cada una cuenta.

Lo que hacemos es **recortarlos** (*capping* o *winsorización*): un valor por encima del límite pasa a valer el límite. La flor se queda, pero su valor extremo pesa menos. `clip` lo hace en una línea, y en test usamos **los límites de train**.

In [ ]:
X_train = X_train.clip(lower=limite_inferior, upper=limite_superior, axis=1)
X_test = X_test.clip(lower=limite_inferior, upper=limite_superior, axis=1)

print("Outliers en train tras recortar:", ((X_train < limite_inferior) | (X_train > limite_superior)).sum().sum())

> 💡 **Los árboles de decisión son robustos a los outliers**: solo comparan "¿es mayor o menor que un umbral?", así que un valor de 4,4 o de 10 cae en la misma rama. El *capping* importa mucho más en modelos como la regresión lineal o KNN, que veremos más adelante. Lo hacemos aquí para practicar el flujo completo.

---
## 7. Estandarización

La estandarización (*z-score*) deja cada variable con **media 0 y desviación típica 1**: $z = \dfrac{x - \text{media}}{\text{desviación}}$. Así ninguna variable domina solo por tener números más grandes.

In [ ]:
escalador = StandardScaler()
X_train_esc = escalador.fit_transform(X_train)   # aprende media y desviación de train
X_test_esc = escalador.transform(X_test)         # aplica las de train

X_train_esc.describe().loc[["mean", "std"]].round(2)

### ¿Le importa la escala a un árbol?

Un árbol pregunta cosas como "¿`largo_petalo` ≤ 2,45?". Estandarizar cambia el umbral (pasaría a ser, por ejemplo, "≤ -0,74"), pero **no cambia el orden de las flores**, así que el árbol hace exactamente las mismas divisiones. Comprobémoslo:

In [ ]:
arbol_sin_escalar = DecisionTreeClassifier(random_state=SEED).fit(X_train, y_train)
arbol_escalado = DecisionTreeClassifier(random_state=SEED).fit(X_train_esc, y_train)

print("Accuracy sin escalar:", accuracy_score(y_test, arbol_sin_escalar.predict(X_test)))
print("Accuracy escalado:   ", accuracy_score(y_test, arbol_escalado.predict(X_test_esc)))

**Mismo resultado.** Aun así, dejamos el `StandardScaler` dentro del `Pipeline`: es una buena costumbre, no estorba, y si cambias el árbol por un modelo sensible a la escala (KNN, regresión logística, SVM) no tendrás que tocar nada más.

---
## 8. ¿Cómo funciona un árbol de decisión?

Un árbol de decisión clasifica haciendo **preguntas de sí/no** sobre las variables, como el juego de "¿Quién es quién?":

```
                ¿largo_petalo ≤ 2,45?
               /                     \
             sí                       no
          setosa              ¿ancho_petalo ≤ 1,75?
                              /                  \
                            sí                    no
                        versicolor            virginica
```

**¿Cómo elige las preguntas?** En cada paso prueba todas las variables y todos los umbrales posibles, y se queda con la pregunta que deja los grupos **más puros** (con flores de una sola especie). La impureza se mide con:

- **Gini** (por defecto): probabilidad de equivocarse si etiquetas al azar según las proporciones del grupo. 0 = grupo puro.
- **Entropía**: mide el "desorden" del grupo. También 0 = grupo puro.

**El problema:** si lo dejamos crecer sin límite, el árbol sigue preguntando hasta que **cada hoja es pura**, aunque tenga que memorizar flores sueltas. Eso es **sobreajuste** (*overfitting*): acierta todo en train y falla con datos nuevos.

### Hiperparámetros principales

Los **hiperparámetros** son las "perillas" del modelo que fijamos **nosotros** antes de entrenar (los **parámetros**, como los umbrales, los aprende el modelo).

| Hiperparámetro | Qué controla | Por defecto |
|---|---|---|
| `criterion` | Medida de impureza: `"gini"` o `"entropy"` | `"gini"` |
| `max_depth` | Número máximo de preguntas seguidas | `None` (sin límite) |
| `min_samples_split` | Mínimo de flores en un nodo para poder dividirlo | `2` |
| `min_samples_leaf` | Mínimo de flores en cada hoja | `1` |

Con los valores por defecto el árbol **no tiene freno**. Limitar la profundidad o exigir hojas más grandes lo hace más simple y, normalmente, mejor con datos nuevos.

---
## 9. Modelo base (parámetros por defecto)

Empezamos siempre con un **modelo base**: el más simple posible, sin ajustar nada. Nos da una referencia para saber si lo que hagamos después mejora de verdad.

Volvemos a partir de `X_train` (ya con outliers recortados). El `Pipeline` encadena **imputar → escalar → árbol** y garantiza que todo se aprende solo con train.

In [ ]:
modelo_base = Pipeline([
    ("imputador", SimpleImputer(strategy="median")),
    ("escalador", StandardScaler()),
    ("arbol", DecisionTreeClassifier(random_state=SEED)),
])

modelo_base.fit(X_train, y_train)

arbol_base = modelo_base.named_steps["arbol"]
print("Profundidad:", arbol_base.get_depth(), "| Hojas:", arbol_base.get_n_leaves())
print("Accuracy en train:", accuracy_score(y_train, modelo_base.predict(X_train)))
print("Accuracy en test: ", accuracy_score(y_test, modelo_base.predict(X_test)))

Fíjate en el **100 % en train**: el árbol creció hasta clasificar perfectamente cada flor de entrenamiento. Es la señal típica de que está **memorizando**.

### 9.1 Validación cruzada

Con solo 30 flores de test, el resultado depende mucho de **qué** flores cayeron en test. La **validación cruzada** (*k-fold*) da una estimación más fiable:

1. Divide train en 5 partes (*folds*).
2. Entrena con 4 y evalúa con la restante.
3. Repite 5 veces, cambiando la parte de evaluación, y promedia.

`StratifiedKFold` mantiene la proporción de especies en cada parte.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

scores_base = cross_val_score(modelo_base, X_train, y_train, cv=cv, scoring="accuracy")
print("Accuracy en cada fold:", scores_base.round(3))
print(f"Media: {scores_base.mean():.3f} ± {scores_base.std():.3f}")

folds = pd.DataFrame({"fold": [f"Fold {i}" for i in range(1, 6)], "accuracy": scores_base})
ax = sns.barplot(data=folds, x="fold", y="accuracy", color="tab:blue")
ax.axhline(scores_base.mean(), color="black", linestyle="--", label=f"Media = {scores_base.mean():.3f}")
ax.bar_label(ax.containers[0], fmt="%.3f")
ax.set_ylim(0.6, 1.05)
ax.set_title("Validación cruzada del modelo base")
ax.legend(loc="lower right")
plt.show()

Cada fold da un resultado distinto según qué flores le tocaron. Por eso informamos siempre **media ± desviación típica**: la desviación nos dice cuánto podemos fiarnos de la media.

---
## 10. Optimización de hiperparámetros

¿Qué valores de `max_depth`, `min_samples_leaf`… funcionan mejor? No hay que adivinarlo: **los probamos**.

`GridSearchCV` prueba **todas las combinaciones** de una rejilla (*grid*) y evalúa cada una con validación cruzada **sobre train**. Al final reentrena la mejor combinación con todo train.

> ⚠️ **El test no se toca durante la búsqueda.** Si eligiéramos los hiperparámetros mirando el test, estaríamos haciendo trampa y la nota final sería demasiado optimista.

Para acceder a un paso del `Pipeline`, el nombre del hiperparámetro lleva delante el nombre del paso y **dos guiones bajos**: `arbol__max_depth`.

In [ ]:
rejilla = {
    "arbol__criterion": ["gini", "entropy"],
    "arbol__max_depth": [2, 3, 4, 5, None],
    "arbol__min_samples_split": [2, 5, 10],
    "arbol__min_samples_leaf": [1, 2, 4, 8],
}

busqueda = GridSearchCV(modelo_base, rejilla, cv=cv, scoring="accuracy", n_jobs=-1)
busqueda.fit(X_train, y_train)

n_combinaciones = len(busqueda.cv_results_["params"])
print(f"Combinaciones probadas: {n_combinaciones} ({n_combinaciones * cv.get_n_splits()} entrenamientos)")
print("Mejores hiperparámetros:", busqueda.best_params_)
print(f"Mejor accuracy en validación cruzada: {busqueda.best_score_:.3f}")

Veamos las mejores combinaciones. Suele haber **empates**: muchas combinaciones distintas dan la misma *accuracy*. En ese caso `GridSearchCV` elige la primera de la rejilla.

In [ ]:
resultados = pd.DataFrame(busqueda.cv_results_)
columnas = ["param_arbol__criterion", "param_arbol__max_depth", "param_arbol__min_samples_split",
            "param_arbol__min_samples_leaf", "mean_test_score", "std_test_score", "rank_test_score"]

resultados[columnas].sort_values("rank_test_score", kind="stable").head(10).round(3)

### 10.1 ¿Cómo afecta cada hiperparámetro?

Un mapa de calor muestra la *accuracy* media de validación para cada combinación de `max_depth` y `min_samples_leaf` (con el mejor `criterion` y `min_samples_split`):

In [ ]:
mejor = busqueda.best_params_
filtro = ((resultados["param_arbol__criterion"] == mejor["arbol__criterion"])
          & (resultados["param_arbol__min_samples_split"] == mejor["arbol__min_samples_split"]))

tabla = (resultados[filtro]
         .assign(max_depth=lambda d: d["param_arbol__max_depth"].map(lambda v: "sin límite" if pd.isna(v) else str(int(v))))
         .pivot_table(index="max_depth", columns="param_arbol__min_samples_leaf", values="mean_test_score"))

sns.heatmap(tabla, annot=True, fmt=".3f", cmap="viridis")
plt.title("Accuracy media en validación cruzada")
plt.xlabel("min_samples_leaf")
plt.ylabel("max_depth")
plt.show()

- Con `max_depth=2` (solo 2 preguntas seguidas) o con hojas de al menos 8 flores, el árbol es **demasiado simple** y empeora: eso es **subajuste** (*underfitting*).
- A partir de `max_depth=3` la *accuracy* deja de mejorar: más preguntas no ayudan a clasificar flores nuevas, solo a memorizar las de train.
- Hay muchos **empates**. Ante un empate, lo sensato es quedarse con el **árbol más simple**, y eso es justo lo que pasa aquí: `max_depth=3` es la primera combinación ganadora de la rejilla.

### 10.2 El modelo optimizado

In [ ]:
modelo_opt = busqueda.best_estimator_   # ya reentrenado con todo train

arbol_opt = modelo_opt.named_steps["arbol"]
print("Profundidad:", arbol_opt.get_depth(), "| Hojas:", arbol_opt.get_n_leaves())
print("Accuracy en train:", accuracy_score(y_train, modelo_opt.predict(X_train)))
print("Accuracy en test: ", accuracy_score(y_test, modelo_opt.predict(X_test)))

### 10.3 Visualizar el árbol

Una gran ventaja de los árboles es que **se pueden leer**. Cada caja muestra la pregunta, la impureza, cuántas flores llegan (`samples`), cuántas hay de cada especie (`value`) y la clase mayoritaria.

> 💡 Los umbrales están en **unidades estandarizadas** (el árbol ve los datos después del `StandardScaler`): por ejemplo, `-0,74` significa "0,74 desviaciones típicas por debajo de la media".

In [ ]:
plt.figure(figsize=(16, 8))
plot_tree(arbol_opt, feature_names=list(X_train.columns), class_names=list(arbol_opt.classes_),
          filled=True, rounded=True, fontsize=10)
plt.title("Árbol optimizado")
plt.show()

### 10.4 Importancia de las variables

El árbol mide cuánto **reduce la impureza** cada variable en total. Cuanto más alta, más útil le ha sido.

In [ ]:
importancias = pd.DataFrame({
    "variable": X_train.columns,
    "importancia": arbol_opt.feature_importances_,
}).sort_values("importancia", ascending=False)

ax = sns.barplot(data=importancias, x="importancia", y="variable", hue="variable", palette="Set2")
ax.set_title("Importancia de las variables (árbol optimizado)")
plt.show()

Confirma lo que vimos en el EDA: **las medidas del pétalo** son las que separan las especies.

### 10.5 Validación cruzada: base vs optimizado

`GridSearchCV` ya usó validación cruzada por dentro, pero solo con una métrica. Con `cross_validate` evaluamos los dos modelos **con los mismos 5 folds** y con varias métricas a la vez. Con `return_train_score=True` también obtenemos el resultado en train, para ver el sobreajuste.

> 💡 El `best_score_` de la búsqueda es algo **optimista**: elegimos la mejor de 120 combinaciones mirando esos mismos folds. Para una estimación sin ese sesgo, la referencia final es el test (o una *validación cruzada anidada*, que veremos más adelante).

In [ ]:
metricas_cv = {"accuracy": "accuracy", "f1_macro": "f1_macro", "auc_ovr": "roc_auc_ovr"}

filas = []
for nombre, modelo in [("Base (por defecto)", modelo_base), ("Optimizado", modelo_opt)]:
    cv_res = cross_validate(modelo, X_train, y_train, cv=cv, scoring=metricas_cv, return_train_score=True)
    for metrica in metricas_cv:
        for conjunto, clave in [("Train", f"train_{metrica}"), ("Validación", f"test_{metrica}")]:
            for fold, valor in enumerate(cv_res[clave], start=1):
                filas.append({"modelo": nombre, "métrica": metrica, "conjunto": conjunto,
                              "fold": fold, "valor": valor})
cv_resultados = pd.DataFrame(filas)

# Resumen: media ± desviación de cada métrica
resumen_cv = (cv_resultados
              .groupby(["modelo", "conjunto", "métrica"])["valor"]
              .agg(media="mean", desv=lambda s: s.std(ddof=0))   # ddof=0, como NumPy en cross_val_score
              .unstack("métrica")
              .round(3))
resumen_cv

In [ ]:
validacion = cv_resultados[cv_resultados["conjunto"] == "Validación"]

g = sns.catplot(data=validacion, x="modelo", y="valor", col="métrica", hue="modelo",
                kind="box", palette="Set2", showfliers=False, height=4, aspect=0.9)
g.map_dataframe(sns.stripplot, x="modelo", y="valor", color="black", size=6, jitter=0.1)
g.set_axis_labels("", "Valor en validación")
g.set_titles("{col_name}")
g.figure.suptitle("Validación cruzada (5 folds): cada punto es un fold", y=1.03)
plt.show()

In [ ]:
accuracy_cv = cv_resultados[cv_resultados["métrica"] == "accuracy"]

ax = sns.barplot(data=accuracy_cv, x="modelo", y="valor", hue="conjunto", palette="Set2", errorbar="sd")
ax.set_ylim(0.8, 1.05)
ax.set_ylabel("Accuracy (media ± desv. típica)")
ax.set_xlabel("")
ax.set_title("Train vs validación: el hueco es el sobreajuste")
ax.legend(loc="lower right")
plt.show()

- En **accuracy y F1** los dos modelos **empatan** en validación: aciertan y fallan las mismas flores en cada fold. Hay un fold claramente más difícil que el resto (el punto aislado de abajo).
- En **AUC** el optimizado es mejor: sus probabilidades ordenan mejor las flores (lo veremos en la sección 13).
- En las **barras**, el modelo base tiene train = 1,0 y un hueco mayor con validación; el optimizado llega a lo mismo en validación **con un árbol más pequeño** y menos sobreajuste.

---
## 11. Evaluación de los modelos

Ahora sí, usamos el **test** para comparar los dos modelos con datos que ninguno ha visto.

### 11.1 Matriz de confusión

Cada fila es la especie **real** y cada columna la **predicha**. La diagonal son los aciertos; fuera de ella, los errores.

In [ ]:
modelos = {"Base (por defecto)": modelo_base, "Optimizado": modelo_opt}
clases = modelo_base.classes_

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for (nombre, modelo), ax in zip(modelos.items(), axes):
    matriz = confusion_matrix(y_test, modelo.predict(X_test), labels=clases)
    sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=clases, yticklabels=clases, ax=ax)
    ax.set_title(nombre)
    ax.set_xlabel("Predicha")
    ax.set_ylabel("Real")
plt.tight_layout()
plt.show()

Como esperábamos, *setosa* nunca se confunde. Los errores, si los hay, están entre *versicolor* y *virginica*.

### 11.2 Classification report

Resume las métricas principales **por clase**:

- **Precision**: de las flores que el modelo llamó *virginica*, ¿qué porcentaje lo era de verdad?
- **Recall** (sensibilidad): de las *virginica* reales, ¿qué porcentaje encontró el modelo?
- **F1-score**: media armónica de precision y recall (equilibra las dos).
- **Support**: cuántas flores de esa clase hay en test.

In [ ]:
for nombre, modelo in modelos.items():
    print(f"===== {nombre} =====")
    print(classification_report(y_test, modelo.predict(X_test), digits=3))

---
## 12. Curvas de aprendizaje

Una **curva de aprendizaje** muestra cómo cambian los resultados en train y en validación a medida que el modelo ve **más datos**. Es la mejor herramienta para diagnosticar:

| Lo que ves | Diagnóstico |
|---|---|
| Train muy alto y validación bastante más baja | **Sobreajuste** (memoriza) |
| Train y validación bajas y juntas | **Subajuste** (demasiado simple) |
| Train y validación altas y juntas | ✅ Buen equilibrio |
| La validación sigue subiendo al final | Más datos probablemente ayudarían |

`learning_curve` entrena el modelo con el 10 %, 20 %… 100 % de train, con validación cruzada en cada tamaño.

In [ ]:
def calcular_curva_aprendizaje(modelo, nombre):
    """Devuelve un DataFrame largo (una fila por tamaño, fold y conjunto) para seaborn."""
    tamaños, scores_train, scores_val = learning_curve(
        modelo, X_train, y_train, cv=cv, scoring="accuracy",
        train_sizes=np.linspace(0.1, 1.0, 8), shuffle=True, random_state=SEED,
    )
    filas = []
    for conjunto, scores in [("Train", scores_train), ("Validación", scores_val)]:
        for tamaño, scores_tamaño in zip(tamaños, scores):
            for score in scores_tamaño:
                filas.append({"modelo": nombre, "conjunto": conjunto, "n_flores": tamaño, "accuracy": score})
    return pd.DataFrame(filas)


curvas = pd.concat([calcular_curva_aprendizaje(m, n) for n, m in modelos.items()])

g = sns.relplot(data=curvas, x="n_flores", y="accuracy", hue="conjunto", col="modelo",
                kind="line", marker="o", errorbar="sd", height=4, aspect=1.3)
g.set_axis_labels("Flores de entrenamiento", "Accuracy")
g.set(ylim=(0.6, 1.02))
plt.show()

La sombra es la **desviación típica** entre los 5 folds: cuanto más ancha, más depende el resultado de qué flores tocaron.

- **Modelo base:** la curva de train está siempre en 1,0: memoriza cada flor que ve. El hueco con validación es el sobreajuste.
- **Modelo optimizado:** train ya no es perfecto (el árbol tiene freno), pero el hueco con validación es menor: **generaliza mejor**.

En un dataset tan fácil como Iris, ambos llegan a una validación parecida. En datos reales, con más ruido, la diferencia suele ser mucho mayor.

---
## 13. Curvas ROC y AUC

Además de la clase, el modelo puede dar una **probabilidad** para cada clase con `predict_proba`. En un árbol, esa probabilidad es la proporción de cada especie en la hoja donde cae la flor.

In [ ]:
probas_opt = modelo_opt.predict_proba(X_test)
pd.DataFrame(probas_opt, columns=clases).round(2).head()

### ¿Qué es la curva ROC?

Para decidir si una flor es *virginica*, elegimos un **umbral** de probabilidad (por ejemplo, 0,5). Al mover el umbral cambian los aciertos y los errores. La curva ROC dibuja, para todos los umbrales posibles:

- **Eje Y — Tasa de verdaderos positivos (TPR, recall):** de las *virginica*, ¿cuántas detecta?
- **Eje X — Tasa de falsos positivos (FPR):** de las que **no** son *virginica*, ¿cuántas marca por error?

El **AUC** (área bajo la curva) lo resume en un número: **1,0 = perfecto**, **0,5 = adivinar al azar** (la diagonal).

La ROC es para problemas de **dos clases**. Con 3 clases usamos **uno contra el resto** (*one-vs-rest*, OvR): una curva por especie ("¿es *virginica* o no?").

In [ ]:
# Convertimos el target en una columna binaria por clase (one-vs-rest)
y_test_bin = label_binarize(y_test, classes=clases)

filas = []
for nombre, modelo in modelos.items():
    probas = modelo.predict_proba(X_test)
    for i, clase in enumerate(clases):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], probas[:, i])
        auc_clase = roc_auc_score(y_test_bin[:, i], probas[:, i])
        for x, y_ in zip(fpr, tpr):
            filas.append({"modelo": nombre, "clase": f"{clase} (AUC = {auc_clase:.3f})", "FPR": x, "TPR": y_})
roc = pd.DataFrame(filas)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for (nombre, _), ax in zip(modelos.items(), axes):
    sns.lineplot(data=roc[roc["modelo"] == nombre], x="FPR", y="TPR", hue="clase",
                 palette="Set2", estimator=None, sort=False, marker="o", ax=ax)
    ax.plot([0, 1], [0, 1], "k--", label="Azar (AUC = 0,5)")
    ax.set_title(f"Curvas ROC · {nombre}")
    ax.set_xlabel("Tasa de falsos positivos (FPR)")
    ax.set_ylabel("Tasa de verdaderos positivos (TPR)")
    ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

Fíjate en la forma de las curvas del **modelo base**: tienen muy pocos puntos. Como sus hojas son puras, sus probabilidades son siempre 0 o 1, así que no hay umbrales intermedios que probar. El árbol optimizado tiene hojas con mezcla de especies y da probabilidades intermedias, lo que suele mejorar el AUC.

El **AUC macro** promedia el AUC de las tres clases:

In [ ]:
for nombre, modelo in modelos.items():
    auc_macro = roc_auc_score(y_test, modelo.predict_proba(X_test), multi_class="ovr", average="macro")
    print(f"{nombre:<20} AUC (OvR, macro) = {auc_macro:.3f}")

---
## 14. Reporte de resultados

Reunimos todas las métricas en una tabla. Es lo que presentarías al final de un proyecto.

In [ ]:
def evaluar(modelo, scores_cv):
    """Métricas principales de un modelo ya entrenado."""
    pred_test = modelo.predict(X_test)
    return {
        "Accuracy train": accuracy_score(y_train, modelo.predict(X_train)),
        "Accuracy CV (media)": scores_cv.mean(),
        "Accuracy CV (desv.)": scores_cv.std(),
        "Accuracy test": accuracy_score(y_test, pred_test),
        "F1 macro test": f1_score(y_test, pred_test, average="macro"),
        "AUC OvR test": roc_auc_score(y_test, modelo.predict_proba(X_test), multi_class="ovr"),
        "Profundidad": modelo.named_steps["arbol"].get_depth(),
        "Hojas": modelo.named_steps["arbol"].get_n_leaves(),
    }


scores_opt = cross_val_score(modelo_opt, X_train, y_train, cv=cv, scoring="accuracy")

reporte = pd.DataFrame({
    "Base (por defecto)": evaluar(modelo_base, scores_base),
    "Optimizado": evaluar(modelo_opt, scores_opt),
}).T

reporte.round(3)

In [ ]:
metricas = ["Accuracy train", "Accuracy CV (media)", "Accuracy test", "F1 macro test", "AUC OvR test"]
reporte_largo = (reporte[metricas]
                 .reset_index(names="modelo")
                 .melt(id_vars="modelo", var_name="métrica", value_name="valor"))

ax = sns.barplot(data=reporte_largo, x="valor", y="métrica", hue="modelo", palette="Set2")
ax.set_xlim(0.8, 1.04)
ax.set_title("Comparación de modelos")
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
for contenedor in ax.containers:
    ax.bar_label(contenedor, fmt="%.3f", fontsize=9, padding=2)
plt.show()

### 📝 Conclusiones

Al leer el reporte, fíjate en:

1. **Sobreajuste del modelo base:** acierta el 100 % en train pero no en validación. El árbol sin límites memoriza.
2. **Misma accuracy en validación cruzada:** los dos modelos empatan. En Iris, limitar el árbol no le hace acertar más flores nuevas…
3. **…pero el optimizado es mejor modelo:** con **menos profundidad y menos hojas** consigue lo mismo, se interpreta mejor, sobreajusta menos y sus probabilidades son más útiles (**mayor AUC**, tanto en validación cruzada como en test). A igualdad de resultados, **gana el modelo más simple**.
4. **Cuidado con el test pequeño:** con 30 flores, 1 error = 3,3 puntos de *accuracy*. La ventaja del optimizado en test es **una sola flor**: no es concluyente por sí sola; la validación cruzada es mejor guía.
5. **Iris es un dataset fácil:** casi cualquier modelo supera el 90 %. Las ventajas de optimizar se notan mucho más con datos reales y ruidosos.

> 🧪 **Prueba tú:** cambia `random_state` en `train_test_split` (por ejemplo, a 0 o 7) y vuelve a ejecutar todo. ¿Cambia cuál modelo gana en test? ¿Y en validación cruzada?

---
## 15. 📋 Chuleta resumen

| Tarea | Código |
|---|---|
| Distribución por clase | `sns.countplot`, `sns.kdeplot(hue=...)`, `sns.pairplot(df, hue=...)` |
| Correlaciones | `sns.heatmap(df.corr(), annot=True)` |
| Nulos y duplicados | `df.isna().sum()`, `df.duplicated().sum()`, `df.drop_duplicates()` |
| Separar con proporciones | `train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)` |
| Recortar outliers | `X.clip(lower=lim_inf, upper=lim_sup, axis=1)` (límites de train) |
| Árbol de decisión | `DecisionTreeClassifier(max_depth=..., min_samples_leaf=..., random_state=42)` |
| Pipeline | `Pipeline([("imputador", ...), ("escalador", ...), ("arbol", ...)])` |
| Validación cruzada | `cross_val_score(modelo, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42))` |
| CV con varias métricas | `cross_validate(modelo, X, y, cv=cv, scoring={...}, return_train_score=True)` |
| Búsqueda de hiperparámetros | `GridSearchCV(pipeline, {"arbol__max_depth": [...]}, cv=cv)` |
| Mejor modelo | `busqueda.best_params_`, `busqueda.best_estimator_` |
| Matriz de confusión | `confusion_matrix(y_test, pred)` + `sns.heatmap` |
| Métricas por clase | `classification_report(y_test, pred)` |
| Curva de aprendizaje | `learning_curve(modelo, X, y, cv=cv, train_sizes=...)` |
| ROC / AUC multiclase | `roc_curve`, `roc_auc_score(y, probas, multi_class="ovr")` |
| Ver el árbol | `plot_tree(arbol, feature_names=..., class_names=..., filled=True)` |

---
### 🎓 ¿Qué sigue?

Ya sabes recorrer el flujo completo: **explorar → limpiar → preparar → entrenar → optimizar → evaluar → reportar**. En las próximas clases usaremos este mismo esquema con otros modelos y datasets más difíciles, donde la optimización marca de verdad la diferencia.